# 02 — Clinical Feature Clustering

Clusters OASIS-1 patients using only the **10 clinical variables** (no MRI features).
A Generalized Gower distance matrix is computed to handle the mix of continuous,
ordinal, and categorical columns, then k-medoids (FasterPAM, k=4) is applied.
CDR is loaded separately and used only for post-hoc evaluation.

**Output:** `clinical_clustering_results.csv` — `patient_id`, `cluster`, `CDR`

In [5]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "robust-mixed-dist", "kmedoids", "scikit-learn", "-q"],
    check=True
)

CompletedProcess(args=['C:\\Users\\aregk\\AppData\\Local\\Programs\\Python\\Python312\\python.exe', '-m', 'pip', 'install', 'robust-mixed-dist', 'kmedoids', 'scikit-learn', '-q'], returncode=0)

In [6]:
import os
import numpy as np
import pandas as pd

from robust_mixed_dist.mixed import generalized_gower_dist_matrix
import kmedoids

## Step 1 — Load data

Load `oasis_combined.csv` and keep only the 10 requested clinical columns plus
`patient_id`. CDR is read separately so it never influences the clustering.

In [7]:
DATA_DIR   = r"C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data"
CSV_PATH   = os.path.join(DATA_DIR, "method_1_HOG_PCA", "oasis_combined.csv")
OUTPUT_CSV = os.path.join(DATA_DIR, "notebooks", "clinical_clustering_results.csv")

CLINICAL_COLS = ["Age", "M/F", "Hand", "Educ", "SES", "MMSE", "eTIV", "nWBV", "ASF", "Delay"]

raw = pd.read_csv(CSV_PATH)
df  = raw[["patient_id"] + CLINICAL_COLS].copy()
cdr = raw[["patient_id", "CDR"]].copy()   # kept aside — evaluation only

print(f"Loaded {df.shape[0]} patients, {df.shape[1]-1} clinical columns")
df.head(3)

Loaded 416 patients, 10 clinical columns


,patient_id,Age,M/F,Hand,Educ,SES,MMSE,eTIV,nWBV,ASF,Delay
0,OAS1_0001,74,F,R,2.0,3.0,29.0,1344,0.743,1.306,NaN
1,OAS1_0002,55,F,R,4.0,1.0,29.0,1147,0.810,1.531,NaN
2,OAS1_0003,73,F,R,4.0,3.0,27.0,1454,0.708,1.207,NaN


## Step 2 — Missing value analysis

Count missingness per column before touching anything, then decide:

| Column | Situation | Action |
|--------|-----------|--------|
| `Delay` | 100 % missing | **Drop** — carries no information |
| `Hand` | Constant (`R` for all patients) | **Drop** — zero variance, contributes nothing to any distance |
| `Educ`, `SES`, `MMSE` | ~43–48 % missing | **Impute** with column median |
| All others | 0 % missing | Keep as-is |

In [8]:
missing    = df.isnull().sum().rename("missing")
pct        = (missing / len(df) * 100).round(1).rename("%")
unique_cnt = df.nunique().rename("unique_vals")

summary = pd.concat([missing, pct, unique_cnt], axis=1)
print(summary.to_string())

            missing      %  unique_vals
patient_id        0    0.0          416
Age               0    0.0           73
M/F               0    0.0            2
Hand              0    0.0            1
Educ            181   43.5            5
SES             200   48.1            5
MMSE            181   43.5           17
eTIV              0    0.0          301
nWBV              0    0.0          182
ASF               0    0.0          275
Delay           416  100.0            0


In [9]:
# Drop uninformative columns
df = df.drop(columns=["Delay", "Hand"])
print("Dropped: Delay (100% missing), Hand (constant 'R')")

# Impute ordinal / continuous columns with median
for col in ["Educ", "SES", "MMSE"]:
    med = df[col].median()
    n_filled = df[col].isnull().sum()
    df[col] = df[col].fillna(med)
    print(f"  {col}: filled {n_filled} NaNs with median = {med}")

print(f"\nRemaining missing values: {df.isnull().sum().sum()}")
print(f"Final feature columns   : {df.columns.drop('patient_id').tolist()}")

Dropped: Delay (100% missing), Hand (constant 'R')
  Educ: filled 181 NaNs with median = 3.0
  SES: filled 200 NaNs with median = 2.0
  MMSE: filled 181 NaNs with median = 29.0

Remaining missing values: 0
Final feature columns   : ['Age', 'M/F', 'Educ', 'SES', 'MMSE', 'eTIV', 'nWBV', 'ASF']


## Step 3 — Generalized Gower distance matrix

`generalized_gower_dist_matrix` from `robust_mixed_dist` handles **mixed variable types**.
Columns must be ordered: **quantitative first, then binary, then multi-class**.

| Type | Columns | Distance used |
|------|---------|---------------|
| Quantitative (`p1`) | Age, Educ, SES, MMSE, eTIV, nWBV, ASF | `minkowski` (Manhattan, q=1) |
| Binary (`p2`) | M/F (encoded 0/1) | `sokal` |
| Multi-class (`p3`) | *(none)* | — |

Result is a symmetric 416×416 matrix with values in [0, 1].

In [10]:
# Encode M/F as binary integer (F=0, M=1)
df["MF_bin"] = df["M/F"].map({"F": 0, "M": 1}).astype(int)

# Column order required by the function: quantitative | binary | multi-class
quant_cols  = ["Age", "Educ", "SES", "MMSE", "eTIV", "nWBV", "ASF"]
binary_cols = ["MF_bin"]

p1 = len(quant_cols)    # 7 quantitative
p2 = len(binary_cols)   # 1 binary
p3 = 0                  # 0 multi-class

X = df[quant_cols + binary_cols].values.astype(float)

print(f"Feature matrix shape : {X.shape}")
print(f"  p1 (quantitative)  : {p1} — {quant_cols}")
print(f"  p2 (binary)        : {p2} — {binary_cols}")
print(f"  p3 (multi-class)   : {p3}")

D = generalized_gower_dist_matrix(
    X,
    p1=p1, p2=p2, p3=p3,
    d1="minkowski",   # Manhattan distance for quantitative
    d2="sokal",       # Sokal distance for binary
    d3="hamming",     # not used (p3=0) but required by signature
    q=1
)

print(f"\nDistance matrix shape : {D.shape}")
print(f"Value range           : [{D.min():.4f}, {D.max():.4f}]")
print(f"Is symmetric          : {np.allclose(D, D.T)}")

Feature matrix shape : (416, 8)
  p1 (quantitative)  : 7 — ['Age', 'Educ', 'SES', 'MMSE', 'eTIV', 'nWBV', 'ASF']
  p2 (binary)        : 1 — ['MF_bin']
  p3 (multi-class)   : 0

Distance matrix shape : (416, 416)
Value range           : [0.0000, 5.5779]
Is symmetric          : True


## Step 4 — K-medoids clustering (FasterPAM, k=4)

`kmedoids.fasterpam` works directly on the precomputed distance matrix.
FasterPAM is significantly faster than classic PAM with the same solution quality.
k=4 aligns with the four CDR levels: 0, 0.5, 1, 2.

In [11]:
K = 4

result = kmedoids.fasterpam(
    D,
    medoids=K,
    random_state=42
)

labels = np.array(result.labels)   # 0-indexed cluster assignments

print(f"Loss (sum of distances to medoids): {result.loss:.4f}")
print(f"Medoid indices: {result.medoids}")
print(f"Medoid patient IDs: {df['patient_id'].iloc[result.medoids].tolist()}")
print()
sizes = pd.Series(labels).value_counts().sort_index().rename("count")
print("Cluster sizes:")
print(sizes.to_string())

Loss (sum of distances to medoids): 202.4246
Medoid indices: [254  65 353  93]
Medoid patient IDs: ['OAS1_0282', 'OAS1_0070', 'OAS1_0389', 'OAS1_0101']

Cluster sizes:
0    128
1    128
2     76
3     84


## Step 5 — Evaluation: cluster vs CDR

CDR was never used during clustering. Merge it back in now to see whether
the clinical clusters align with dementia severity (CDR 0=none, 0.5=very mild,
1=mild, 2=moderate). Patients without CDR are excluded from the crosstab.

In [12]:
results = df[["patient_id"]].copy()
results["cluster"] = labels
results = results.merge(cdr, on="patient_id", how="left")

print(f"Patients with CDR    : {results['CDR'].notnull().sum()}")
print(f"Patients without CDR : {results['CDR'].isnull().sum()}")
print()

has_cdr = results[results["CDR"].notnull()].copy()
has_cdr["CDR"] = has_cdr["CDR"].astype(str)

ct = pd.crosstab(
    has_cdr["cluster"],
    has_cdr["CDR"],
    margins=True,
    margins_name="Total"
)
print("Cluster × CDR (counts, patients with CDR only):")
ct

Patients with CDR    : 235
Patients without CDR : 181

Cluster × CDR (counts, patients with CDR only):


CDR,0.0,0.5,1.0,2.0,Total
cluster,,,,,
0,35,17,11,0,63
1,62,22,8,1,93
2,21,14,5,0,40
3,17,17,4,1,39
Total,135,70,28,2,235


In [13]:
# Row-normalised: CDR distribution within each cluster
ct_norm = pd.crosstab(
    has_cdr["cluster"],
    has_cdr["CDR"],
    normalize="index"
).round(3)

print("Row-normalised (proportion of each CDR level within each cluster):")
ct_norm

Row-normalised (proportion of each CDR level within each cluster):


CDR,0.0,0.5,1.0,2.0
cluster,,,,
0,0.556,0.270,0.175,0.000
1,0.667,0.237,0.086,0.011
2,0.525,0.350,0.125,0.000
3,0.436,0.436,0.103,0.026


## Step 6 — Save results

Save `patient_id`, `cluster`, and `CDR` for all 416 patients.
CDR is `NaN` for patients who had no clinical label in the original dataset.

In [14]:
results.to_csv(OUTPUT_CSV, index=False)

print(f"Saved : {OUTPUT_CSV}")
print(f"Shape : {results.shape}  (rows=patients, cols=patient_id, cluster, CDR)")
print()
print(results.head(5).to_string(index=False))

Saved : C:\Users\aregk\OneDrive\Desktop\Thesis_downoading_the_data\notebooks\clinical_clustering_results.csv
Shape : (416, 3)  (rows=patients, cols=patient_id, cluster, CDR)

patient_id  cluster  CDR
 OAS1_0001        1  0.0
 OAS1_0002        1  0.0
 OAS1_0003        0  0.5
 OAS1_0004        3  NaN
 OAS1_0005        2  NaN
